In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load your data
df = pd.read_csv('data/raw/upi_transactions.csv',
                 parse_dates=['timestamp'])
print(f"Loaded: {df.shape}")
print(df.dtypes)

Loaded: (10000, 8)
transaction_id               object
user_id                      object
timestamp            datetime64[ns]
amount                      float64
merchant_category            object
city_tier                    object
payment_status               object
device_type                  object
dtype: object


In [3]:
# Null check
print("=== NULL VALUES ===")
print(df.isnull().sum())

# Duplicate check
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")

# Basic shape
print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Unique users: {df['user_id'].nunique()}")
print(f"Payment status split:{df['payment_status'].value_counts()}")

=== NULL VALUES ===
transaction_id       0
user_id              0
timestamp            0
amount               0
merchant_category    0
city_tier            0
payment_status       0
device_type          0
dtype: int64
Duplicate rows: 0
Date range: 2024-01-01 00:39:00 → 2024-12-30 23:41:00
Unique users: 500
Payment status split:payment_status
Success    8786
Failed      926
Pending     288
Name: count, dtype: int64


In [4]:
# Successful transactions only (for spending analysis)
df_ok = df[df['payment_status'] == 'Success'].copy()
print(f"Successful txns: {len(df_ok):,} of {len(df):,}")
print(f"Success rate: {len(df_ok)/len(df)*100:.1f}%")

Successful txns: 8,786 of 10,000
Success rate: 87.9%


In [5]:
df_ok['hour']        = df_ok['timestamp'].dt.hour
df_ok['day_name']    = df_ok['timestamp'].dt.day_name()
df_ok['month']       = df_ok['timestamp'].dt.month
df_ok['month_name']  = df_ok['timestamp'].dt.strftime('%b')
df_ok['week']        = df_ok['timestamp'].dt.isocalendar().week.astype(int)
df_ok['is_weekend']  = df_ok['timestamp'].dt.dayofweek >= 5
df_ok['quarter']     = df_ok['timestamp'].dt.quarter

# Time of day buckets
def time_bucket(h):
    if h < 6:   return 'Late Night'
    if h < 12:  return 'Morning'
    if h < 17:  return 'Afternoon'
    if h < 21:  return 'Evening'
    return 'Night'

df_ok['time_of_day'] = df_ok['hour'].apply(time_bucket)
print(df_ok[['timestamp','hour','day_name','time_of_day']].head(3))

            timestamp  hour  day_name time_of_day
0 2024-11-23 03:01:00     3  Saturday  Late Night
1 2024-10-06 02:37:00     2    Sunday  Late Night
2 2024-10-14 06:45:00     6    Monday     Morning


In [7]:
df_ok['amount_bucket'] = pd.cut(
    df_ok['amount'],
    bins=[0, 100, 500, 2000, 10000, 50000],
    labels=['Micro(<₹100)', 'Small(₹100-500)',
            'Mid(₹500-2K)', 'Large(₹2K-10K)',
            'Very Large(>₹10K)']
)
print(df_ok['amount_bucket'].value_counts())

amount_bucket
Mid(₹500-2K)         3639
Small(₹100-500)      3088
Large(₹2K-10K)       1469
Micro(<₹100)          486
Very Large(>₹10K)     104
Name: count, dtype: int64


In [8]:
os.makedirs('data/cleaned', exist_ok=True)
df_ok.to_csv('data/cleaned/upi_clean.csv', index=False)
df.to_csv('data/cleaned/upi_all_statuses.csv', index=False)

print("Saved:")
print(f"  data/cleaned/upi_clean.csv       ({len(df_ok):,} rows)")
print(f"  data/cleaned/upi_all_statuses.csv ({len(df):,} rows)")

Saved:
  data/cleaned/upi_clean.csv       (8,786 rows)
  data/cleaned/upi_all_statuses.csv (10,000 rows)
